# 04 - Evaluation

Stage 4 of 4. Reads the predictions written by `03_model_training.ipynb` and produces
every metric table, comparison chart and conclusion in the project.

### Three analyses

1. **Imbalanced vs Balanced training** - does balancing the training set help? (Experiments A vs B)
2. **Overfitting analysis** - train vs test gap, and why balanced models behave differently.
3. **Supervised vs Unsupervised** - do anomaly detectors compete with labelled classifiers?

**Which metrics matter for fraud?**
- **Recall** -> how many real frauds we catch (missing fraud is costly)
- **Precision** -> how many fraud alerts are real (false alarms annoy customers)
- **F1** -> balance of precision & recall
- **PR-AUC** -> best single score for rare-event problems (better than accuracy)
- **Accuracy** -> misleading here (predicting "not fraud" always looks ~99% accurate)

## 1. Imports & setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
from pathlib import Path

KAGGLE_IN = Path("/kaggle/input")


def find_data_dir():
    """Folder holding fraudTrain.csv / fraudTest.csv (Kaggle input mount or local archive/)."""
    for c in [KAGGLE_IN / "fraud-detection", Path("archive"), Path(".")]:
        if (c / "fraudTrain.csv").exists():
            return c
    for p in sorted(KAGGLE_IN.rglob("fraudTrain.csv")) if KAGGLE_IN.exists() else []:
        return p.parent
    return Path("archive")


# Raw CSVs: Kaggle dataset mount when on Kaggle, else the local archive/ folder.
DATA_DIR = find_data_dir()

# Everything this project generates goes into output/.
OUT = Path("/kaggle/working/output") if Path("/kaggle/working").exists() else Path("output")
for sub in ["data", "plots", "models", "preds", "results"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)


def upstream(rel):
    """Locate a file written by an earlier notebook (local run or Kaggle kernel input)."""
    local = OUT / rel
    if local.exists():
        return local
    if KAGGLE_IN.exists():
        for p in sorted(KAGGLE_IN.rglob(rel.split("/")[-1])):
            if p.as_posix().endswith("output/" + rel):
                return p
    raise FileNotFoundError("Run the earlier notebook first - missing: " + rel)


def save_fig(name):
    """Save the current matplotlib figure into output/plots/."""
    plt.savefig(OUT / "plots" / (name + ".png"), dpi=150, bbox_inches="tight")


print("DATA_DIR:", DATA_DIR)
print("OUT     :", OUT)

## 2. Load predictions from notebook 03

In [ ]:
test_pred = pd.read_parquet(upstream("preds/test_preds.parquet"))
train_sample_pred = pd.read_parquet(upstream("preds/train_sample_preds.parquet"))
train_bal_pred = pd.read_parquet(upstream("preds/train_bal_preds.parquet"))
rf_importance = pd.read_csv(upstream("models/rf_feature_importance.csv"), index_col=0)["importance"]

y_test = test_pred["y_true"]

# Group A and B were also the supervised arm of the supervised-vs-unsupervised study,
# so the same fitted models appear under two names.
ALIAS = {
    "Supervised | Logistic Regression": "Imbalanced | Logistic Regression",
    "Supervised | Random Forest": "Imbalanced | Random Forest",
}


def get(name, kind):
    """Prediction ('pred') or score ('score') column for a model on the test set."""
    return test_pred[ALIAS.get(name, name) + "::" + kind].values


print("Test rows:", len(y_test), "| fraud in test:", int(y_test.sum()))
print("Models with stored test predictions:")
for c in test_pred.columns:
    if c.endswith("::pred"):
        print("  -", c[:-6])

## 3. Scoring helpers

In [ ]:
def evaluate_model(name, y_te, y_pred, y_prob):
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_prob),
        "PR-AUC": average_precision_score(y_te, y_prob),
    }

    print("=" * 60)
    print(name)
    print("=" * 60)
    print(classification_report(y_te, y_pred, digits=4))
    print("Confusion matrix [[TN FP], [FN TP]]:")
    print(confusion_matrix(y_te, y_pred))
    return metrics, y_pred, y_prob


def evaluate(name, y_te, y_pred, y_score):
    metrics = {
        "Model": name,
        "Type": "Unsupervised" if name.startswith("Unsupervised") else "Supervised",
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_score),
        "PR-AUC": average_precision_score(y_te, y_score),
    }
    print("=" * 60)
    print(name)
    print(classification_report(y_te, y_pred, digits=4))
    print("Confusion [[TN FP],[FN TP]]:\n", confusion_matrix(y_te, y_pred))
    return metrics

# Part 1 - Imbalanced vs Balanced training

Same two models, same feature matrix, same imbalanced test set. Only the **training**
distribution changes.

In [ ]:
results = []
predictions = {}  # store probs for curves

for name in [
    "Imbalanced | Logistic Regression",
    "Imbalanced | Random Forest",
    "Balanced | Logistic Regression",
    "Balanced | Random Forest",
]:
    m, pred, prob = evaluate_model(name, y_test, get(name, "pred"), get(name, "score"))
    results.append(m)
    predictions[name] = prob

## 4. Comparison table

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.set_index("Model")
display_df = results_df.copy()
for col in display_df.columns:
    display_df[col] = display_df[col].map(lambda x: f"{x:.4f}")

print("Comparison on the SAME imbalanced test set")
display_df

In [ ]:
plot_df = results_df.reset_index().melt(id_vars="Model", var_name="Metric", value_name="Score")
focus = plot_df[plot_df["Metric"].isin(["Precision", "Recall", "F1", "PR-AUC", "ROC-AUC"])]

plt.figure(figsize=(12, 5))
sns.barplot(data=focus, x="Metric", y="Score", hue="Model")
plt.title("Imbalanced vs Balanced training - key metrics (same test set)")
plt.ylim(0, 1.05)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_fig("08_balancing_key_metrics")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for name, prob in predictions.items():
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=axes[0])
axes[0].set_title("ROC curves")

for name, prob in predictions.items():
    PrecisionRecallDisplay.from_predictions(y_test, prob, name=name, ax=axes[1])
axes[1].set_title("Precision-Recall curves (more important for fraud)")

plt.tight_layout()
save_fig("09_balancing_roc_pr_curves")
plt.show()

In [ ]:
# Confusion matrices for Random Forest (usually stronger of the two)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

cm_imb = confusion_matrix(y_test, get("Imbalanced | Random Forest", "pred"))
cm_bal = confusion_matrix(y_test, get("Balanced | Random Forest", "pred"))

sns.heatmap(cm_imb, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Pred 0", "Pred 1"], yticklabels=["Actual 0", "Actual 1"])
axes[0].set_title("RF - trained on Imbalanced")

sns.heatmap(cm_bal, annot=True, fmt="d", cmap="Oranges", ax=axes[1],
            xticklabels=["Pred 0", "Pred 1"], yticklabels=["Actual 0", "Actual 1"])
axes[1].set_title("RF - trained on Balanced")

plt.tight_layout()
save_fig("10_balancing_confusion_matrices")
plt.show()

In [ ]:
# Feature importance from Random Forest (imbalanced train)
imp = rf_importance.sort_values(ascending=False).head(12)

plt.figure(figsize=(9, 5))
sns.barplot(x=imp.values, y=imp.index, color="#4C78A8")
plt.title("Top features (Random Forest, imbalanced train)")
plt.xlabel("Importance")
plt.tight_layout()
save_fig("11_rf_feature_importance")
plt.show()

## 5. Thesis answer - Do we need a balanced dataset?

### How to read the results

1. **If balanced training gives higher Recall but much lower Precision**
   -> model catches more fraud, but raises many false alarms. Common with under-sampling.

2. **If imbalanced training gives higher Precision / PR-AUC / F1**
   -> keeping the real class ratio often generalizes better to the real (rare-fraud) world.

3. **Accuracy alone is not enough**
   -> with ~99.6% non-fraud, a useless model can look "accurate".

### Conclusion for this dataset

- **Balancing is not always required**, and it is **not automatically better**.
- Balancing (under-sampling) often **increases Recall** (finds more fraud) but can **hurt Precision** (more false positives) because the model sees fraud as "common" during training, while in the test world fraud is still rare.
- Training on the **original imbalanced** data usually matches reality better, especially for **PR-AUC** and **Precision**.

> **Use the original imbalanced data as the main training setup** if the goal is realistic performance on rare fraud.
> **Use balancing as a comparison / sensitivity study**, or when the business priority is maximum fraud catch-rate (Recall) and the bank can accept more false alerts.
> Best practice: compare both, report Precision, Recall, F1, and PR-AUC, and choose based on the business cost of missed fraud vs false alarms - not based on Accuracy.

In [ ]:
# Auto summary to help you write the thesis conclusion
summary = results_df.copy()
print("Best by PR-AUC :", summary["PR-AUC"].idxmax(), f"({summary['PR-AUC'].max():.4f})")
print("Best by F1     :", summary["F1"].idxmax(), f"({summary['F1'].max():.4f})")
print("Best by Recall :", summary["Recall"].idxmax(), f"({summary['Recall'].max():.4f})")
print("Best by Precision:", summary["Precision"].idxmax(), f"({summary['Precision'].max():.4f})")

print("\n--- Short interpretation tip ---")
best_pr = summary["PR-AUC"].idxmax()
if "Imbalanced" in best_pr:
    print("PR-AUC prefers IMBALANCED training -> keeping real class ratio looks better for rare fraud ranking.")
else:
    print("PR-AUC prefers BALANCED training -> under-sampling helped overall rare-event ranking on this run.")

best_rec = summary["Recall"].idxmax()
if "Balanced" in best_rec:
    print("Recall prefers BALANCED training -> useful if catching fraud matters more than false alarms.")

# Part 2 - Overfitting analysis

Each model is scored on:
- **Train** - the data it was fitted on (100k sample for the imbalanced models; the full 1:1 set for the balanced models)
- **Test** - the same original imbalanced `fraudTest` for all four models

| Interpretation | Condition |
|---|---|
| **Not overfitting** | Train and test F1 / PR-AUC are close (small gap) |
| **Overfitting** | Train metrics near-perfect, test metrics much lower (large gap) |
| **Prior mismatch (balanced)** | Good train scores on 1:1 data, poor Precision/F1 on imbalanced test - not memorization, but wrong fraud rate learned |

In [ ]:
def split_metrics(name, split, y, pred, prob):
    return {
        "Model": name,
        "Split": split,
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "PR-AUC": average_precision_score(y, prob),
    }


y_tr = train_sample_pred["y_true"]
y_bal = train_bal_pred["y_true"]
print("Imbalanced TRAIN sample for scoring:", len(y_tr), "| fraud in sample:", int(y_tr.sum()))
print("Balanced TRAIN set for scoring    :", len(y_bal), "| fraud in set   :", int(y_bal.sum()))

SHORT = {
    "Imbalanced | LR": "Imbalanced | Logistic Regression",
    "Imbalanced | RF": "Imbalanced | Random Forest",
    "Balanced | LR": "Balanced | Logistic Regression",
    "Balanced | RF": "Balanced | Random Forest",
}

ov_rows = []
for short in ["Imbalanced | LR", "Imbalanced | RF"]:
    ov_rows.append(split_metrics(short, "train", y_tr,
                                 train_sample_pred[short + "::pred"], train_sample_pred[short + "::score"]))
    ov_rows.append(split_metrics(short, "test", y_test,
                                 get(SHORT[short], "pred"), get(SHORT[short], "score")))
for short in ["Balanced | LR", "Balanced | RF"]:
    ov_rows.append(split_metrics(short, "train", y_bal,
                                 train_bal_pred[short + "::pred"], train_bal_pred[short + "::score"]))
    ov_rows.append(split_metrics(short, "test", y_test,
                                 get(SHORT[short], "pred"), get(SHORT[short], "score")))

ov = pd.DataFrame(ov_rows)
print("\nTrain vs Test performance")
display(ov.pivot(index="Model", columns="Split", values=["Precision", "Recall", "F1", "PR-AUC"]).round(4))

for metric in ["F1", "PR-AUC"]:
    gap = ov.pivot(index="Model", columns="Split", values=metric)
    gap[f"{metric} gap (train - test)"] = gap["train"] - gap["test"]
    print(f"\n{metric} gap (train minus test; large positive gap suggests overfitting):")
    print(gap.round(4))

In [ ]:
# Graph 1: train vs test - Precision, Recall, F1, PR-AUC
plot_ov = ov.melt(
    id_vars=["Model", "Split"],
    value_vars=["Precision", "Recall", "F1", "PR-AUC"],
    var_name="Metric",
    value_name="Score",
)
g = sns.catplot(
    data=plot_ov, x="Metric", y="Score", hue="Split",
    col="Model", kind="bar", height=3.4, aspect=0.95, col_wrap=2,
)
g.set(ylim=(0, 1.05))
g.fig.suptitle("Train vs test - overfitting check (same models)", y=1.03)
g.savefig(OUT / "plots" / "12_train_vs_test.png", dpi=150, bbox_inches="tight")
plt.show()

# Graph 2: test-set comparison (imbalanced vs balanced training)
test_only = ov[ov["Split"] == "test"].copy()
focus = test_only.melt(
    id_vars="Model",
    value_vars=["Precision", "Recall", "F1", "PR-AUC"],
    var_name="Metric",
    value_name="Score",
)
plt.figure(figsize=(10, 5))
sns.barplot(data=focus, x="Metric", y="Score", hue="Model")
plt.title("Test set only - imbalanced vs balanced training")
plt.ylim(0, 1.05)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_fig("13_test_only_comparison")
plt.show()

## 6. Conclusion - overfitting and balanced vs imbalanced accuracy

**1. Data validation** (see `01_data_loading.ipynb`)
- Train: ~1.29M transactions, fraud rate ~**0.58%**. Test: ~555k transactions, fraud rate ~**0.39%**.
- No `trans_num` overlap between train and test -> no direct row leakage.
- Balanced training uses **7,506 fraud + 7,506 non-fraud** (50/50). Test remains highly imbalanced (~0.4% fraud). Models trained on balanced data are evaluated on a distribution they never saw during training.

**2. Is the result overfitting?**

| Model | Overfitting? | Evidence |
|---|---|---|
| **Imbalanced \| Random Forest** | **No** | Train and test **F1** and **PR-AUC** stay close. The model generalizes to the imbalanced test set. |
| **Imbalanced \| Logistic Regression** | **No** | Small train-test gap; low test Recall is underfitting the minority class, not memorization. |
| **Balanced \| Random Forest** | **Not classic overfitting** | Train scores on 1:1 data look strong, but test **Precision** (~0.15) and **F1** (~0.26) drop sharply. The model learned that fraud is **common** (50% in train) while it is **rare** in test (0.4%). This is a **class-prior mismatch**, not memorization of training rows. |
| **Balanced \| Logistic Regression** | Same pattern | High train performance on balanced data; poor test Precision because of false alarms on the real prior. |

**Overfitting** = large train - test gap on **F1 / PR-AUC** with the model fitting noise in training rows. **Prior mismatch** = good scores on balanced train, poor Precision on imbalanced test because the decision threshold is wrong for real-world fraud rate. The balanced RF result is mainly **prior mismatch**, not overfitting.

**3. Why does balanced training look "more accurate" on some metrics?**

- **Test Accuracy:** imbalanced RF (**~0.999**) > balanced RF (**~0.979**) > balanced LR (**~0.90**). Balanced training does **not** achieve higher test Accuracy.
- **Test Recall:** balanced RF is highest (**~0.97** vs **~0.75** for imbalanced RF) - it catches more fraud but at the cost of many false positives.
- **Test Precision / F1 / PR-AUC:** imbalanced RF wins (**Precision ~0.94, F1 ~0.84, PR-AUC ~0.88**). These are the correct metrics for rare fraud.
- High **Accuracy** on imbalanced data is dominated by correctly classifying the majority (non-fraud) class. It should not be used as the main metric.

**Final answer**

The imbalanced Random Forest model is **not overfitting**: train and test F1/PR-AUC are consistent, and it gives the best fraud-detection trade-off on the real test set. Balanced training increases **Recall** but reduces **Precision** and overall **F1/PR-AUC** on the imbalanced test set because the model is trained under an artificial 50/50 fraud rate. For deployment on this dataset, **imbalanced training with Random Forest** is the appropriate choice; balancing is useful only when maximum Recall is explicitly required despite heavy false-alarm cost.

# Part 3 - Supervised vs Unsupervised

**Question:** When labels exist, do supervised classifiers beat unsupervised anomaly detectors?

| Type | Models | Uses fraud labels at fit? |
|---|---|---|
| **Supervised** | Logistic Regression, Random Forest | Yes (imbalanced train) |
| **Unsupervised** | Isolation Forest, One-Class SVM, LOF | No - fit on **non-fraud only** |

Same engineered features and the **same imbalanced test set** for every model.

In [ ]:
results_su, scores, preds = [], {}, {}

for name in [
    "Supervised | Logistic Regression",
    "Supervised | Random Forest",
    "Unsupervised | Isolation Forest",
    "Unsupervised | One-Class SVM",
    "Unsupervised | Local Outlier Factor",
]:
    pred, score = get(name, "pred"), get(name, "score")
    m = evaluate(name, y_test, pred, score)
    results_su.append(m); scores[m["Model"]] = score; preds[m["Model"]] = pred

## 7. Comparison table

In [ ]:
results_su_df = pd.DataFrame(results_su).set_index("Model")
display_su = results_su_df.copy()
for c in display_su.columns:
    if c != "Type":
        display_su[c] = display_su[c].map(lambda x: f"{x:.4f}")

print("Supervised vs Unsupervised - SAME imbalanced test set")
display_su

## 8. Graphs

In [ ]:
# Metric bar chart
plot_su = results_su_df.reset_index().melt(
    id_vars=["Model", "Type"],
    value_vars=["Precision", "Recall", "F1", "PR-AUC"],
    var_name="Metric",
    value_name="Score",
)
plt.figure(figsize=(11, 5))
sns.barplot(data=plot_su, x="Metric", y="Score", hue="Model")
plt.title("Supervised vs Unsupervised - key metrics (same test set)")
plt.ylim(0, 1.05)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_fig("14_supervised_vs_unsupervised_metrics")
plt.show()

In [ ]:
# ROC + Precision-Recall curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, score in scores.items():
    RocCurveDisplay.from_predictions(y_test, score, name=name, ax=axes[0])
axes[0].set_title("ROC curves")
for name, score in scores.items():
    PrecisionRecallDisplay.from_predictions(y_test, score, name=name, ax=axes[1])
axes[1].set_title("Precision-Recall curves (more important for fraud)")
plt.tight_layout()
save_fig("15_supervised_vs_unsupervised_curves")
plt.show()

In [ ]:
# Confusion: best supervised (RF) vs best unsupervised by PR-AUC
best_unsup = results_su_df.loc[results_su_df["Type"] == "Unsupervised", "PR-AUC"].idxmax()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name, cmap in [
    (axes[0], "Supervised | Random Forest", "Blues"),
    (axes[1], best_unsup, "Oranges"),
]:
    cm = confusion_matrix(y_test, preds[name])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap=cmap, ax=ax,
        xticklabels=["Pred 0", "Pred 1"], yticklabels=["Actual 0", "Actual 1"],
    )
    ax.set_title(name)
plt.tight_layout()
save_fig("16_supervised_vs_unsupervised_confusion")
plt.show()
print("Best unsupervised by PR-AUC:", best_unsup)

In [ ]:
# Grouped view: Supervised vs Unsupervised average on key metrics
grp = results_su_df.groupby("Type")[["Precision", "Recall", "F1", "PR-AUC"]].mean()
grp.T.plot(kind="bar", figsize=(8, 4), color=["#4C78A8", "#E45756"], rot=0)
plt.title("Mean metrics by approach (Supervised vs Unsupervised)")
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.legend(title="Type")
plt.tight_layout()
save_fig("17_supervised_vs_unsupervised_grouped")
plt.show()
grp.round(4)

## 9. Discussion

1. **Supervised Random Forest wins** on Precision, F1, and PR-AUC.
2. **Unsupervised methods are weaker** - they flag "unusual," not "fraud." Rare legitimate spend looks anomalous -> many false alarms (low Precision / PR-AUC).
3. **Use unsupervised** when labels are missing or delayed; **use supervised** once labels exist.
4. **Takeaway:** labels matter. Balancing experiments (Part 1) are a supervised design choice; anomaly detection does not replace supervised learning here.

In [ ]:
print("Best PR-AUC :", results_su_df["PR-AUC"].idxmax(), f"({results_su_df['PR-AUC'].max():.4f})")
print("Best F1     :", results_su_df["F1"].idxmax(), f"({results_su_df['F1'].max():.4f})")
print("Best Precision:", results_su_df["Precision"].idxmax(), f"({results_su_df['Precision'].max():.4f})")
print("Best Recall :", results_su_df["Recall"].idxmax(), f"({results_su_df['Recall'].max():.4f})")

## 10. Save all result tables

In [ ]:
results_df.round(6).to_csv(OUT / "results" / "imbalanced_vs_balanced_results.csv")
ov.round(6).to_csv(OUT / "results" / "overfitting_train_vs_test.csv", index=False)
results_su_df.round(6).to_csv(OUT / "results" / "supervised_vs_unsupervised_results.csv")
grp.round(6).to_csv(OUT / "results" / "supervised_vs_unsupervised_grouped.csv")

print("Result tables:")
for p in sorted((OUT / "results").iterdir()):
    print("  results/" + p.name)
print("\nPlots:")
for p in sorted((OUT / "plots").iterdir()):
    print("  plots/" + p.name)